# Bakehouse Franchises & Sales

**Dataset:** `samples.bakehouse.sales_franchises`, `samples.bakehouse.sales_transactions`

**Difficulty:** Medium

**Topics:** join, groupBy, window, ranking

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window as W

franchises = spark.read.table("samples.bakehouse.sales_franchises")
transactions = spark.read.table("samples.bakehouse.sales_transactions")

## Problem 1

Join franchises with transactions on `franchiseID`. Compute total revenue and transaction count per franchise.

**Expected output columns:**
- `franchiseID`
- `name`
- `country`
- `total_revenue`
- `transaction_count`

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = (
    transactions
    .groupBy("franchiseID")
    .agg(
        F.sum("totalPrice").alias("total_revenue"),
        F.count("transactionID").alias("transaction_count")
    )
    .join(franchises, "franchiseID")
    .select(
        "franchiseID",
        "name",
        "country",
        "total_revenue",
        "transaction_count"
    )
    .orderBy(F.col("total_revenue").desc())
)

result_1.show()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'country' in cols, "Missing column: country"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rev = result_1.agg(F.min('total_revenue')).collect()[0][0]
assert min_rev >= 0, f"Expected total_revenue >= 0, found min={min_rev}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Find the top 5 franchises by total revenue.

**Expected output columns:**
- `franchiseID`
- `name`
- `country`
- `size`
- `total_revenue`

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = (
    transactions
    .groupBy("franchiseID")
    .agg(
        F.sum("totalPrice").alias("total_revenue")
    )
    .join(franchises, "franchiseID")
    .select(
        "franchiseID",
        "name",
        "country",
        "size",
        "total_revenue"
    )
    .orderBy(F.col("total_revenue").desc())
    .limit(5)
)

result_2.show()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'country' in cols, "Missing column: country"
assert 'size' in cols, "Missing column: size"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 5, f"Expected at most 5 rows (top 5), got {cnt}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Compute total revenue per franchise size (e.g., small/medium/large), along with the count of franchises in each size category.

**Expected output columns:**
- `size`
- `total_revenue`
- `franchise_count`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = (
    transactions
    .join(franchises, "franchiseID")
    .groupBy("size")
    .agg(
        F.sum("totalPrice").alias("total_revenue"),
        F.countDistinct("franchiseID").alias("franchise_count")
    )
    .orderBy(F.col("total_revenue").desc())
)

result_3.show()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'size' in cols, "Missing column: size"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'franchise_count' in cols, "Missing column: franchise_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_fc = result_3.agg(F.min('franchise_count')).collect()[0][0]
assert min_fc >= 1, f"Expected franchise_count >= 1, found min={min_fc}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Using a window function, rank franchises by revenue within each country. Keep only rank <= 3.

**Expected output columns:**
- `country`
- `name`
- `total_revenue`
- `rank`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4
w = W.partitionBy("country").orderBy(F.col("total_revenue").desc())
result_4 = (
    transactions
    .join(franchises, "franchiseID")
    .groupBy("country", "name")
    .agg(F.sum("totalPrice").alias("total_revenue"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") <= 3)
    .orderBy("country", "rank")
)

result_4.show()

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'country' in cols, "Missing column: country"
assert 'name' in cols, "Missing column: name"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'rank' in cols, "Missing column: rank"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_rank = result_4.agg(F.max('rank')).collect()[0][0]
assert max_rank <= 3, f"Expected rank <= 3, found max={max_rank}"
min_rank = result_4.agg(F.min('rank')).collect()[0][0]
assert min_rank >= 1, f"Expected rank >= 1, found min={min_rank}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find the best-selling product per franchise (top 1 product by total quantity sold).

**Expected output columns:**
- `franchiseID`
- `name`
- `best_product`
- `total_quantity`

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5
w = W.partitionBy("franchiseID").orderBy(F.col("total_quantity").desc())
result_5 = (
    transactions
    .groupBy("franchiseID", F.col("product").alias("best_product"))
    .agg(F.sum("quantity").alias("total_quantity"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") == 1)
    .join(franchises, "franchiseID")
    .select("franchiseID", "name", "best_product", "total_quantity")
)

result_5.show()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'best_product' in cols, "Missing column: best_product"
assert 'total_quantity' in cols, "Missing column: total_quantity"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_qty = result_5.agg(F.min('total_quantity')).collect()[0][0]
assert min_qty >= 0, f"Expected total_quantity >= 0, found min={min_qty}"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Compute average transaction value per franchise. Flag franchises above the overall average with a boolean column `above_overall_avg`.

**Expected output columns:**
- `franchiseID`
- `name`
- `avg_transaction_value`
- `above_overall_avg`

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6
overall_avg = transactions.agg(F.avg("totalPrice").alias("overall_avg")).collect()[0][0]
result_6 = (
    transactions
    .groupBy("franchiseID")
    .agg(F.avg("totalPrice").alias("avg_transaction_value"))
    .withColumn(
        "above_overall_avg",
        F.col("avg_transaction_value") > overall_avg
    )
    .join(franchises, "franchiseID")
    .select("franchiseID", "name", "avg_transaction_value", "above_overall_avg")
    .orderBy(F.col("above_overall_avg").desc())
)

result_6.show()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'avg_transaction_value' in cols, "Missing column: avg_transaction_value"
assert 'above_overall_avg' in cols, "Missing column: above_overall_avg"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
above_cnt = result_6.filter(F.col('above_overall_avg') == True).count()
assert above_cnt > 0, "Expected at least some franchises above overall average"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

Find franchises that have sold every unique product available in the transactions table.
Show `products_sold` and `total_products`, keeping only rows where `products_sold = total_products`.

**Expected output columns:**
- `franchiseID`
- `name`
- `products_sold`
- `total_products`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7
total_products = transactions.select("product").distinct().count()

result_7 = (
    transactions
    .groupBy("franchiseID")
    .agg(F.countDistinct("product").alias("products_sold"))
    .withColumn("total_products", F.lit(total_products))
    .filter(F.col("products_sold") == F.col("total_products"))
    .join(franchises, "franchiseID")
    .select("franchiseID", "name", "products_sold", "total_products")
)

result_7.show()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'name' in cols, "Missing column: name"
assert 'products_sold' in cols, "Missing column: products_sold"
assert 'total_products' in cols, "Missing column: total_products"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
if cnt > 0:
    invalid = result_7.filter(F.col('products_sold') != F.col('total_products')).count()
    assert invalid == 0, f"Found {invalid} rows where products_sold != total_products"
print(f"Problem 7 passed ✓  ({cnt} rows)")